In [25]:
import mysql.connector
from sqlalchemy import create_engine
import pandas as pd

In [26]:

# ETL configuration

CURRENT_PERIOD = 1
MAX_PERIOD = 37
BASE_PERIOD_START = pd.Timestamp('2005-07-01')
RESET_WAREHOUSE = True 

if not (1 <= CURRENT_PERIOD <= MAX_PERIOD):
    raise ValueError('CURRENT_PERIOD must be between 1 and 37.')

period_start = BASE_PERIOD_START + pd.DateOffset(months=CURRENT_PERIOD - 1)
period_end = period_start + pd.DateOffset(months=1)

print('Current period      :', CURRENT_PERIOD)
print('Period start date   :', period_start.date())
print('Period end date     :', period_end.date())
print('Warehouse reset     :', RESET_WAREHOUSE)


Current period      : 1
Period start date   : 2005-07-01
Period end date     : 2005-08-01
Warehouse reset     : True


In [27]:
# Database connections

db_adventureworks2012 = mysql.connector.connect(
    host='localhost',
    user='root',
    passwd='',
    database='adventureworks2012'
)

db_datawarehouse = mysql.connector.connect(
    host='localhost',
    user='root',
    passwd='',
    database='datawarehouse'
)

source_engine = create_engine(
    'mysql+mysqlconnector://root:@localhost:3306/adventureworks2012',
    echo=False
)

dw_engine = create_engine(
    'mysql+mysqlconnector://root:@localhost:3306/datawarehouse',
    echo=False
)

In [28]:
# Housekeeping and warehouse table creation

cursor = db_datawarehouse.cursor()

if RESET_WAREHOUSE:
    cursor.execute('DROP TABLE IF EXISTS fact_store_sales;')
    cursor.execute('DROP TABLE IF EXISTS raw_store_sales;')
    cursor.execute('DROP TABLE IF EXISTS dim_salesperson;')
    cursor.execute('DROP TABLE IF EXISTS dim_time;')
    cursor.execute('DROP TABLE IF EXISTS dim_product;')
    cursor.execute('DROP TABLE IF EXISTS dim_store_customer;')

cursor.execute('''
CREATE TABLE IF NOT EXISTS dim_store_customer (
    customer_key INT AUTO_INCREMENT PRIMARY KEY,
    customer_id INT NOT NULL,
    store_id INT NOT NULL,
    store_name VARCHAR(100),
    account_number VARCHAR(30),
    territory_id INT,
    store_salesperson_id INT NULL,
    UNIQUE KEY uk_dim_store_customer_customer_id (customer_id)
);
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS dim_product (
    product_key INT AUTO_INCREMENT PRIMARY KEY,
    product_id INT NOT NULL,
    product_name VARCHAR(100),
    product_number VARCHAR(50),
    color VARCHAR(30),
    size VARCHAR(10),
    size_unit_measure_code CHAR(6),
    weight_unit_measure_code CHAR(6),
    weight DECIMAL(8,2),
    standard_cost DECIMAL(19,4),
    list_price DECIMAL(19,4),
    product_model_name VARCHAR(100),
    product_subcategory_name VARCHAR(100),
    product_category_name VARCHAR(100),
    sell_start_date DATETIME NULL,
    sell_end_date DATETIME NULL,
    discontinued_date DATETIME NULL,
    UNIQUE KEY uk_dim_product_product_id (product_id)
);
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS dim_time (
    time_key INT PRIMARY KEY,
    full_date DATE NOT NULL,
    day_of_month TINYINT,
    month_num TINYINT,
    month_name VARCHAR(20),
    quarter_num TINYINT,
    year_num SMALLINT,
    week_of_year TINYINT,
    day_name VARCHAR(20),
    is_weekend TINYINT,
    UNIQUE KEY uk_dim_time_full_date (full_date)
);
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS dim_salesperson (
    salesperson_key INT AUTO_INCREMENT PRIMARY KEY,
    salesperson_id INT NOT NULL,
    full_name VARCHAR(250),
    title VARCHAR(16),
    sales_quota DECIMAL(19,4) NULL,
    bonus DECIMAL(19,4) NULL,
    commission_pct DECIMAL(10,4) NULL,
    sales_ytd DECIMAL(19,4) NULL,
    sales_last_year DECIMAL(19,4) NULL,
    UNIQUE KEY uk_dim_salesperson_salesperson_id (salesperson_id)
);
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS raw_store_sales (
    raw_store_sales_id BIGINT AUTO_INCREMENT PRIMARY KEY,
    sales_order_id INT NOT NULL,
    sales_order_detail_id INT NOT NULL,
    order_date DATETIME NOT NULL,
    due_date DATETIME NULL,
    ship_date DATETIME NULL,
    status TINYINT,
    sales_order_number VARCHAR(50),
    purchase_order_number VARCHAR(50),
    account_number VARCHAR(30),
    customer_id INT NOT NULL,
    store_id INT NOT NULL,
    store_name VARCHAR(100),
    salesperson_id INT NULL,
    territory_id INT NULL,
    product_id INT NOT NULL,
    order_qty SMALLINT,
    unit_price DECIMAL(19,4),
    unit_price_discount DECIMAL(19,4),
    line_total DECIMAL(38,6),
    subtotal DECIMAL(19,4),
    tax_amt DECIMAL(19,4),
    freight DECIMAL(19,4),
    total_due DECIMAL(19,4),
    currency_rate_id INT NULL,
    ship_method_id INT NULL,
    comment VARCHAR(256),
    UNIQUE KEY uk_raw_store_sales (sales_order_id, sales_order_detail_id)
);
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS fact_store_sales (
    fact_store_sales_key BIGINT AUTO_INCREMENT PRIMARY KEY,
    sales_order_id INT NOT NULL,
    sales_order_detail_id INT NOT NULL,
    sales_order_number VARCHAR(50),
    customer_key INT NOT NULL,
    product_key INT NOT NULL,
    order_date_key INT NOT NULL,
    salesperson_key INT NOT NULL,
    order_qty SMALLINT NOT NULL,
    unit_price DECIMAL(19,4) NOT NULL,
    unit_price_discount DECIMAL(19,4) NOT NULL,
    gross_amount DECIMAL(19,4) NOT NULL,
    discount_amount DECIMAL(19,4) NOT NULL,
    margin_amount DECIMAL(38,6),
    net_amount DECIMAL(38,6) NOT NULL,
    header_subtotal DECIMAL(19,4),
    header_tax_amt DECIMAL(19,4),
    header_freight DECIMAL(19,4),
    header_total_due DECIMAL(19,4),
    UNIQUE KEY uk_fact_store_sales (sales_order_id, sales_order_detail_id),
    CONSTRAINT fk_fact_customer FOREIGN KEY (customer_key) REFERENCES dim_store_customer(customer_key),
    CONSTRAINT fk_fact_product FOREIGN KEY (product_key) REFERENCES dim_product(product_key),
    CONSTRAINT fk_fact_time FOREIGN KEY (order_date_key) REFERENCES dim_time(time_key),
    CONSTRAINT fk_fact_salesperson FOREIGN KEY (salesperson_key) REFERENCES dim_salesperson(salesperson_key)
);
''')

cursor.execute('''
INSERT IGNORE INTO dim_salesperson
    (salesperson_id, full_name, title, sales_quota, bonus, commission_pct, sales_ytd, sales_last_year)
VALUES
    (-1, 'Unknown', NULL, NULL, NULL, NULL, NULL, NULL);
''')

db_datawarehouse.commit()
cursor.close()
print('Warehouse tables are ready.')

Warehouse tables are ready.


## 1. Extract and load only new store customer dimension rows for the current monthly period

In [29]:

str_sql = f'''
SELECT DISTINCT
    c.CustomerID AS customer_id,
    c.StoreID AS store_id,
    st.Name AS store_name,
    c.AccountNumber AS account_number,
    c.TerritoryID AS territory_id,
    st.SalesPersonID AS store_salesperson_id
FROM adventureworks2012.salesorderheader soh
JOIN adventureworks2012.customer c
    ON soh.CustomerID = c.CustomerID
JOIN adventureworks2012.store st
    ON c.StoreID = st.BusinessEntityID
LEFT JOIN datawarehouse.dim_store_customer d
    ON c.CustomerID = d.customer_id
WHERE c.StoreID IS NOT NULL
  AND soh.OrderDate >= '{period_start.strftime("%Y-%m-%d")}'
  AND soh.OrderDate < '{period_end.strftime("%Y-%m-%d")}'
  AND d.customer_id IS NULL
ORDER BY c.CustomerID ASC;
'''
df_store_customer = pd.read_sql(sql=str_sql, con=db_datawarehouse)
print('New store customers to load:', len(df_store_customer))
df_store_customer.head()


New store customers to load: 38


/tmp/ipykernel_48882/4275712667.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_store_customer = pd.read_sql(sql=str_sql, con=db_datawarehouse)


,customer_id,store_id,store_name,account_number,territory_id,store_salesperson_id
0,29489,302,Area Bike Accessories,AW00029489,4,281
1,29491,306,Clamps & Brackets Co.,AW00029491,5,275
2,29497,318,Great Bikes,AW00029497,1,283
3,29533,392,Small Cycle Store,AW00029533,3,275
4,29549,440,Juvenile Sports Equipment,AW00029549,5,279


In [30]:

if len(df_store_customer) > 0:
    cursor = db_datawarehouse.cursor()
    for _, row in df_store_customer.iterrows():
        # Convert NaN to None for MySQL NULL compatibility
        row_values = tuple(None if pd.isna(val) else val for val in row)
        cursor.execute('''
            INSERT INTO dim_store_customer 
            (customer_id, store_id, store_name, account_number, territory_id, store_salesperson_id)
            VALUES (%s, %s, %s, %s, %s, %s)
        ''', row_values)
    db_datawarehouse.commit()
    cursor.close()

pd.read_sql('SELECT * FROM dim_store_customer ORDER BY customer_key LIMIT 10;', con=db_datawarehouse)


/tmp/ipykernel_48882/1077284544.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql('SELECT * FROM dim_store_customer ORDER BY customer_key LIMIT 10;', con=db_datawarehouse)


,customer_key,customer_id,store_id,store_name,account_number,territory_id,store_salesperson_id
0,1,29489,302,Area Bike Accessories,AW00029489,4,281
1,2,29491,306,Clamps & Brackets Co.,AW00029491,5,275
2,3,29497,318,Great Bikes,AW00029497,1,283
3,4,29533,392,Small Cycle Store,AW00029533,3,275
4,5,29549,440,Juvenile Sports Equipment,AW00029549,5,279
5,6,29565,484,World Bike Discount Store,AW00029565,4,276
6,7,29566,488,Historic Bicycle Sales,AW00029566,3,275
7,8,29570,496,Grease and Oil Products Company,AW00029570,4,277
8,9,29580,518,Latest Sports Equipment,AW00029580,1,283
9,10,29596,552,Requisite Part Supply,AW00029596,6,282


## 2. Extract and load only new product dimension rows referenced by store sales in the current period

In [31]:

str_sql = f'''
SELECT DISTINCT
    p.ProductID AS product_id,
    p.Name AS product_name,
    p.ProductNumber AS product_number,
    p.Color AS color,
    p.Size AS size,
    p.SizeUnitMeasureCode AS size_unit_measure_code,
    p.WeightUnitMeasureCode AS weight_unit_measure_code,
    p.Weight AS weight,
    p.StandardCost AS standard_cost,
    p.ListPrice AS list_price,
    pm.Name AS product_model_name,
    psc.Name AS product_subcategory_name,
    pc.Name AS product_category_name,
    p.SellStartDate AS sell_start_date,
    p.SellEndDate AS sell_end_date,
    p.DiscontinuedDate AS discontinued_date
FROM adventureworks2012.salesorderheader soh
JOIN adventureworks2012.salesorderdetail sod
    ON soh.SalesOrderID = sod.SalesOrderID
JOIN adventureworks2012.customer c
    ON soh.CustomerID = c.CustomerID
JOIN adventureworks2012.product p
    ON sod.ProductID = p.ProductID
LEFT JOIN adventureworks2012.productmodel pm
    ON p.ProductModelID = pm.ProductModelID
LEFT JOIN adventureworks2012.productsubcategory psc
    ON p.ProductSubcategoryID = psc.ProductSubcategoryID
LEFT JOIN adventureworks2012.productcategory pc
    ON psc.ProductCategoryID = pc.ProductCategoryID
LEFT JOIN datawarehouse.dim_product d
    ON p.ProductID = d.product_id
WHERE c.StoreID IS NOT NULL
  AND soh.OrderDate >= '{period_start.strftime("%Y-%m-%d")}'
  AND soh.OrderDate < '{period_end.strftime("%Y-%m-%d")}'
  AND d.product_id IS NULL
ORDER BY p.ProductID ASC;
'''
df_product = pd.read_sql(sql=str_sql, con=db_datawarehouse)
print('New products to load:', len(df_product))
df_product.head()


New products to load: 46


/tmp/ipykernel_48882/327983966.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_product = pd.read_sql(sql=str_sql, con=db_datawarehouse)


,product_id,product_name,product_number,color,size,size_unit_measure_code,weight_unit_measure_code,weight,standard_cost,list_price,product_model_name,product_subcategory_name,product_category_name,sell_start_date,sell_end_date,discontinued_date
0,707,"Sport-100 Helmet, Red",HL-U509-R,Red,NaN,NaN,NaN,NaN,13.0863,34.99,Sport-100,Helmets,Accessories,2005-07-01,NaT,None
1,708,"Sport-100 Helmet, Black",HL-U509,Black,NaN,NaN,NaN,NaN,13.0863,34.99,Sport-100,Helmets,Accessories,2005-07-01,NaT,None
2,709,"Mountain Bike Socks, M",SO-B909-M,White,M,NaN,NaN,NaN,3.3963,9.50,Mountain Bike Socks,Socks,Clothing,2005-07-01,2006-06-30,None
3,710,"Mountain Bike Socks, L",SO-B909-L,White,L,NaN,NaN,NaN,3.3963,9.50,Mountain Bike Socks,Socks,Clothing,2005-07-01,2006-06-30,None
4,711,"Sport-100 Helmet, Blue",HL-U509-B,Blue,NaN,NaN,NaN,NaN,13.0863,34.99,Sport-100,Helmets,Accessories,2005-07-01,NaT,None


In [32]:

if len(df_product) > 0:
    cursor = db_datawarehouse.cursor()
    for _, row in df_product.iterrows():
        # Convert NaN to None for MySQL NULL compatibility
        row_values = tuple(None if pd.isna(val) else val for val in row)
        cursor.execute('''
            INSERT INTO dim_product 
            (product_id, product_name, product_number, color, size, size_unit_measure_code, 
             weight_unit_measure_code, weight, standard_cost, list_price, product_model_name, 
             product_subcategory_name, product_category_name, sell_start_date, sell_end_date, discontinued_date)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        ''', row_values)
    db_datawarehouse.commit()
    cursor.close()

pd.read_sql('SELECT * FROM dim_product ORDER BY product_key LIMIT 10;', con=db_datawarehouse)


/tmp/ipykernel_48882/1550245762.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql('SELECT * FROM dim_product ORDER BY product_key LIMIT 10;', con=db_datawarehouse)


,product_key,product_id,product_name,product_number,color,size,size_unit_measure_code,weight_unit_measure_code,weight,standard_cost,list_price,product_model_name,product_subcategory_name,product_category_name,sell_start_date,sell_end_date,discontinued_date
0,1,707,"Sport-100 Helmet, Red",HL-U509-R,Red,NaN,NaN,NaN,NaN,13.0863,34.99,Sport-100,Helmets,Accessories,2005-07-01,NaT,None
1,2,708,"Sport-100 Helmet, Black",HL-U509,Black,NaN,NaN,NaN,NaN,13.0863,34.99,Sport-100,Helmets,Accessories,2005-07-01,NaT,None
2,3,709,"Mountain Bike Socks, M",SO-B909-M,White,M,NaN,NaN,NaN,3.3963,9.50,Mountain Bike Socks,Socks,Clothing,2005-07-01,2006-06-30,None
3,4,710,"Mountain Bike Socks, L",SO-B909-L,White,L,NaN,NaN,NaN,3.3963,9.50,Mountain Bike Socks,Socks,Clothing,2005-07-01,2006-06-30,None
4,5,711,"Sport-100 Helmet, Blue",HL-U509-B,Blue,NaN,NaN,NaN,NaN,13.0863,34.99,Sport-100,Helmets,Accessories,2005-07-01,NaT,None
5,6,712,AWC Logo Cap,CA-1098,Multi,NaN,NaN,NaN,NaN,6.9223,8.99,Cycling Cap,Caps,Clothing,2005-07-01,NaT,None
6,7,714,"Long-Sleeve Logo Jersey, M",LJ-0192-M,Multi,M,NaN,NaN,NaN,38.4923,49.99,Long-Sleeve Logo Jersey,Jerseys,Clothing,2005-07-01,NaT,None
7,8,715,"Long-Sleeve Logo Jersey, L",LJ-0192-L,Multi,L,NaN,NaN,NaN,38.4923,49.99,Long-Sleeve Logo Jersey,Jerseys,Clothing,2005-07-01,NaT,None
8,9,716,"Long-Sleeve Logo Jersey, XL",LJ-0192-X,Multi,XL,NaN,NaN,NaN,38.4923,49.99,Long-Sleeve Logo Jersey,Jerseys,Clothing,2005-07-01,NaT,None
9,10,722,"LL Road Frame - Black, 58",FR-R38B-58,Black,58,CM,LB,2.46,204.6251,337.22,LL Road Frame,Road Frames,Components,2005-07-01,NaT,None


## 3. Extract and load only new time dimension rows for store orders in the current period

In [33]:

str_sql = f'''
SELECT DISTINCT
    CAST(DATE_FORMAT(DATE(soh.OrderDate), '%Y%m%d') AS UNSIGNED) AS time_key,
    DATE(soh.OrderDate) AS full_date,
    DAY(DATE(soh.OrderDate)) AS day_of_month,
    MONTH(DATE(soh.OrderDate)) AS month_num,
    MONTHNAME(DATE(soh.OrderDate)) AS month_name,
    QUARTER(DATE(soh.OrderDate)) AS quarter_num,
    YEAR(DATE(soh.OrderDate)) AS year_num,
    WEEK(DATE(soh.OrderDate), 3) AS week_of_year,
    DAYNAME(DATE(soh.OrderDate)) AS day_name,
    CASE
        WHEN DAYOFWEEK(DATE(soh.OrderDate)) IN (1, 7) THEN 1
        ELSE 0
    END AS is_weekend
FROM adventureworks2012.salesorderheader soh
JOIN adventureworks2012.customer c
    ON soh.CustomerID = c.CustomerID
LEFT JOIN datawarehouse.dim_time d
    ON DATE(soh.OrderDate) = d.full_date
WHERE c.StoreID IS NOT NULL
  AND soh.OrderDate >= '{period_start.strftime("%Y-%m-%d")}'
  AND soh.OrderDate < '{period_end.strftime("%Y-%m-%d")}'
  AND d.full_date IS NULL
ORDER BY full_date ASC;
'''
df_time = pd.read_sql(sql=str_sql, con=db_datawarehouse)
print('New dates to load:', len(df_time))
df_time.head()


New dates to load: 1


/tmp/ipykernel_48882/187612036.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_time = pd.read_sql(sql=str_sql, con=db_datawarehouse)


,time_key,full_date,day_of_month,month_num,month_name,quarter_num,year_num,week_of_year,day_name,is_weekend
0,20050701,2005-07-01,1,7,July,3,2005,26,Friday,0


In [34]:
if len(df_time) > 0:
    cursor = db_datawarehouse.cursor()
    for _, row in df_time.iterrows():
        # Convert NaN to None for MySQL NULL compatibility
        row_values = tuple(None if pd.isna(val) else val for val in row)
        cursor.execute('''
            INSERT INTO dim_time 
            (time_key, full_date, day_of_month, month_num, month_name, quarter_num, year_num, week_of_year, day_name, is_weekend)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        ''', row_values)
    db_datawarehouse.commit()
    cursor.close()

pd.read_sql('SELECT * FROM dim_time ORDER BY full_date LIMIT 10;', con=db_datawarehouse)


/tmp/ipykernel_48882/2628401784.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql('SELECT * FROM dim_time ORDER BY full_date LIMIT 10;', con=db_datawarehouse)


,time_key,full_date,day_of_month,month_num,month_name,quarter_num,year_num,week_of_year,day_name,is_weekend
0,20050701,2005-07-01,1,7,July,3,2005,26,Friday,0


## 4. Extract and load only new salesperson dimension rows for the current period

In [35]:

str_sql = f'''
SELECT DISTINCT
    COALESCE(soh.SalesPersonID, st.SalesPersonID) AS salesperson_id,
    TRIM(
        CONCAT(
            COALESCE(p.FirstName, ''),
            CASE WHEN p.MiddleName IS NOT NULL THEN CONCAT(' ', p.MiddleName) ELSE '' END,
            CASE WHEN p.LastName IS NOT NULL THEN CONCAT(' ', p.LastName) ELSE '' END
        )
    ) AS full_name,
    p.Title AS title,
    sp.SalesQuota AS sales_quota,
    sp.Bonus AS bonus,
    sp.CommissionPct AS commission_pct,
    sp.SalesYTD AS sales_ytd,
    sp.SalesLastYear AS sales_last_year
FROM adventureworks2012.salesorderheader soh
JOIN adventureworks2012.customer c
    ON soh.CustomerID = c.CustomerID
JOIN adventureworks2012.store st
    ON c.StoreID = st.BusinessEntityID
LEFT JOIN adventureworks2012.salesperson sp
    ON sp.BusinessEntityID = COALESCE(soh.SalesPersonID, st.SalesPersonID)
LEFT JOIN adventureworks2012.person p
    ON p.BusinessEntityID = sp.BusinessEntityID
LEFT JOIN datawarehouse.dim_salesperson d
    ON COALESCE(soh.SalesPersonID, st.SalesPersonID) = d.salesperson_id
WHERE c.StoreID IS NOT NULL
  AND soh.OrderDate >= '{period_start.strftime("%Y-%m-%d")}'
  AND soh.OrderDate < '{period_end.strftime("%Y-%m-%d")}'
  AND COALESCE(soh.SalesPersonID, st.SalesPersonID) IS NOT NULL
  AND d.salesperson_id IS NULL
ORDER BY salesperson_id ASC;
'''
df_salesperson = pd.read_sql(sql=str_sql, con=db_datawarehouse)
print('New salespersons to load:', len(df_salesperson))
df_salesperson.head()


New salespersons to load: 9


/tmp/ipykernel_48882/305231475.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_salesperson = pd.read_sql(sql=str_sql, con=db_datawarehouse)


,salesperson_id,full_name,title,sales_quota,bonus,commission_pct,sales_ytd,sales_last_year
0,275,Michael G Blythe,None,300000.0,4100.0,0.012,3.763178e+06,1.750406e+06
1,276,Linda C Mitchell,None,250000.0,2000.0,0.015,4.251369e+06,1.439156e+06
2,277,Jillian Carson,None,250000.0,2500.0,0.015,3.189418e+06,1.997186e+06
3,278,Garrett R Vargas,None,250000.0,500.0,0.010,1.453719e+06,1.620277e+06
4,279,Tsvi Michael Reiter,None,300000.0,6700.0,0.010,2.315186e+06,1.849641e+06


In [36]:
if len(df_salesperson) > 0:
    cursor = db_datawarehouse.cursor()
    for _, row in df_salesperson.iterrows():
        # Convert NaN to None for MySQL NULL compatibility
        row_values = tuple(None if pd.isna(val) else val for val in row)
        cursor.execute('''
            INSERT IGNORE INTO dim_salesperson 
            (salesperson_id, full_name, title, sales_quota, bonus, commission_pct, sales_ytd, sales_last_year)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        ''', row_values)
    db_datawarehouse.commit()
    cursor.close()


pd.read_sql('SELECT * FROM dim_salesperson ORDER BY salesperson_key LIMIT 10;', con=db_datawarehouse)


/tmp/ipykernel_48882/4058538777.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql('SELECT * FROM dim_salesperson ORDER BY salesperson_key LIMIT 10;', con=db_datawarehouse)


,salesperson_key,salesperson_id,full_name,title,sales_quota,bonus,commission_pct,sales_ytd,sales_last_year
0,1,-1,Unknown,None,NaN,NaN,NaN,NaN,NaN
1,2,275,Michael G Blythe,None,300000.0,4100.0,0.012,3.763178e+06,1.750406e+06
2,3,276,Linda C Mitchell,None,250000.0,2000.0,0.015,4.251369e+06,1.439156e+06
3,4,277,Jillian Carson,None,250000.0,2500.0,0.015,3.189418e+06,1.997186e+06
4,5,278,Garrett R Vargas,None,250000.0,500.0,0.010,1.453719e+06,1.620277e+06
5,6,279,Tsvi Michael Reiter,None,300000.0,6700.0,0.010,2.315186e+06,1.849641e+06
6,7,280,Pamela O Ansman-Wolfe,None,250000.0,5000.0,0.010,1.352577e+06,1.927059e+06
7,8,281,Shu K Ito,None,250000.0,3550.0,0.010,2.458536e+06,2.073506e+06
8,9,282,José Edvaldo Saraiva,None,250000.0,5000.0,0.015,2.604541e+06,2.038235e+06
9,10,283,David R Campbell,None,250000.0,3500.0,0.012,1.573013e+06,1.371635e+06


## 5. Extract and load only new raw store sales rows for the current period

In [37]:

str_sql = f'''
SELECT
    soh.SalesOrderID AS sales_order_id,
    sod.SalesOrderDetailID AS sales_order_detail_id,
    soh.OrderDate AS order_date,
    soh.DueDate AS due_date,
    soh.ShipDate AS ship_date,
    soh.Status AS status,
    soh.SalesOrderNumber AS sales_order_number,
    soh.PurchaseOrderNumber AS purchase_order_number,
    soh.AccountNumber AS account_number,
    soh.CustomerID AS customer_id,
    c.StoreID AS store_id,
    st.Name AS store_name,
    COALESCE(soh.SalesPersonID, st.SalesPersonID) AS salesperson_id,
    c.TerritoryID AS territory_id,
    sod.ProductID AS product_id,
    sod.OrderQty AS order_qty,
    sod.UnitPrice AS unit_price,
    sod.UnitPriceDiscount AS unit_price_discount,
    sod.LineTotal AS line_total,
    soh.SubTotal AS subtotal,
    soh.TaxAmt AS tax_amt,
    soh.Freight AS freight,
    soh.TotalDue AS total_due,
    soh.CurrencyRateID AS currency_rate_id,
    soh.ShipMethodID AS ship_method_id,
    soh.Comment AS comment
FROM adventureworks2012.salesorderheader soh
JOIN adventureworks2012.salesorderdetail sod
    ON soh.SalesOrderID = sod.SalesOrderID
JOIN adventureworks2012.customer c
    ON soh.CustomerID = c.CustomerID
JOIN adventureworks2012.store st
    ON c.StoreID = st.BusinessEntityID
LEFT JOIN datawarehouse.raw_store_sales r
    ON soh.SalesOrderID = r.sales_order_id
   AND sod.SalesOrderDetailID = r.sales_order_detail_id
WHERE c.StoreID IS NOT NULL
  AND soh.OrderDate >= '{period_start.strftime("%Y-%m-%d")}'
  AND soh.OrderDate < '{period_end.strftime("%Y-%m-%d")}'
  AND r.sales_order_id IS NULL
ORDER BY soh.SalesOrderID, sod.SalesOrderDetailID ASC;
'''
df_raw = pd.read_sql(sql=str_sql, con=db_datawarehouse)
print('New raw store sales rows to load:', len(df_raw))
df_raw.head()


New raw store sales rows to load: 352


/tmp/ipykernel_48882/2392771249.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_raw = pd.read_sql(sql=str_sql, con=db_datawarehouse)


,sales_order_id,sales_order_detail_id,order_date,due_date,ship_date,status,sales_order_number,purchase_order_number,account_number,customer_id,...,unit_price,unit_price_discount,line_total,subtotal,tax_amt,freight,total_due,currency_rate_id,ship_method_id,comment
0,43659,1,2005-07-01,2005-07-13,2005-07-08,5,SO43659,PO522145787,10-4020-000676,29825,...,2024.994,0.0,2024.994,20565.6206,1971.5149,616.0984,23153.2339,NaN,5,None
1,43659,2,2005-07-01,2005-07-13,2005-07-08,5,SO43659,PO522145787,10-4020-000676,29825,...,2024.994,0.0,6074.982,20565.6206,1971.5149,616.0984,23153.2339,NaN,5,None
2,43659,3,2005-07-01,2005-07-13,2005-07-08,5,SO43659,PO522145787,10-4020-000676,29825,...,2024.994,0.0,2024.994,20565.6206,1971.5149,616.0984,23153.2339,NaN,5,None
3,43659,4,2005-07-01,2005-07-13,2005-07-08,5,SO43659,PO522145787,10-4020-000676,29825,...,2039.994,0.0,2039.994,20565.6206,1971.5149,616.0984,23153.2339,NaN,5,None
4,43659,5,2005-07-01,2005-07-13,2005-07-08,5,SO43659,PO522145787,10-4020-000676,29825,...,2039.994,0.0,2039.994,20565.6206,1971.5149,616.0984,23153.2339,NaN,5,None


In [38]:
if len(df_raw) > 0:
    cursor = db_datawarehouse.cursor()
    for _, row in df_raw.iterrows():
        # Convert NaN to None for MySQL NULL compatibility
        row_values = tuple(None if pd.isna(val) else val for val in row)
        cursor.execute('''
    INSERT INTO raw_store_sales 
    (sales_order_id, sales_order_detail_id, order_date, due_date, ship_date, status, 
     sales_order_number, purchase_order_number, account_number, customer_id, store_id, 
     store_name, salesperson_id, territory_id, product_id, order_qty, unit_price, 
     unit_price_discount, line_total, subtotal, tax_amt, freight, total_due, 
     currency_rate_id, ship_method_id, comment)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
''', row_values)
    db_datawarehouse.commit()
    cursor.close()



pd.read_sql('SELECT * FROM raw_store_sales ORDER BY raw_store_sales_id LIMIT 10;', con=db_datawarehouse)


/tmp/ipykernel_48882/2520167132.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql('SELECT * FROM raw_store_sales ORDER BY raw_store_sales_id LIMIT 10;', con=db_datawarehouse)


,raw_store_sales_id,sales_order_id,sales_order_detail_id,order_date,due_date,ship_date,status,sales_order_number,purchase_order_number,account_number,...,unit_price,unit_price_discount,line_total,subtotal,tax_amt,freight,total_due,currency_rate_id,ship_method_id,comment
0,1,43659,1,2005-07-01,2005-07-13,2005-07-08,5,SO43659,PO522145787,10-4020-000676,...,2024.9940,0.0,2024.9940,20565.6206,1971.5149,616.0984,23153.2339,None,5,None
1,2,43659,2,2005-07-01,2005-07-13,2005-07-08,5,SO43659,PO522145787,10-4020-000676,...,2024.9940,0.0,6074.9820,20565.6206,1971.5149,616.0984,23153.2339,None,5,None
2,3,43659,3,2005-07-01,2005-07-13,2005-07-08,5,SO43659,PO522145787,10-4020-000676,...,2024.9940,0.0,2024.9940,20565.6206,1971.5149,616.0984,23153.2339,None,5,None
3,4,43659,4,2005-07-01,2005-07-13,2005-07-08,5,SO43659,PO522145787,10-4020-000676,...,2039.9940,0.0,2039.9940,20565.6206,1971.5149,616.0984,23153.2339,None,5,None
4,5,43659,5,2005-07-01,2005-07-13,2005-07-08,5,SO43659,PO522145787,10-4020-000676,...,2039.9940,0.0,2039.9940,20565.6206,1971.5149,616.0984,23153.2339,None,5,None
5,6,43659,6,2005-07-01,2005-07-13,2005-07-08,5,SO43659,PO522145787,10-4020-000676,...,2039.9940,0.0,4079.9880,20565.6206,1971.5149,616.0984,23153.2339,None,5,None
6,7,43659,7,2005-07-01,2005-07-13,2005-07-08,5,SO43659,PO522145787,10-4020-000676,...,2039.9940,0.0,2039.9940,20565.6206,1971.5149,616.0984,23153.2339,None,5,None
7,8,43659,8,2005-07-01,2005-07-13,2005-07-08,5,SO43659,PO522145787,10-4020-000676,...,28.8404,0.0,86.5212,20565.6206,1971.5149,616.0984,23153.2339,None,5,None
8,9,43659,9,2005-07-01,2005-07-13,2005-07-08,5,SO43659,PO522145787,10-4020-000676,...,28.8404,0.0,28.8404,20565.6206,1971.5149,616.0984,23153.2339,None,5,None
9,10,43659,10,2005-07-01,2005-07-13,2005-07-08,5,SO43659,PO522145787,10-4020-000676,...,5.7000,0.0,34.2000,20565.6206,1971.5149,616.0984,23153.2339,None,5,None


## 6. Transform raw store sales into the fact table using dimension surrogate keys

In [39]:
str_sql = f'''
SELECT
    r.sales_order_id,
    r.sales_order_detail_id,
    r.sales_order_number,
    r.customer_id,
    r.product_id,
    COALESCE(r.salesperson_id, -1) AS salesperson_id,
    DATE(r.order_date) AS order_date,
    DATE(r.ship_date) AS ship_date,
    r.order_qty,
    r.unit_price,
    r.unit_price_discount,
    r.line_total,
    r.subtotal,
    r.tax_amt,
    r.freight,
    r.total_due,
    p.standard_cost
FROM datawarehouse.raw_store_sales r
LEFT JOIN datawarehouse.fact_store_sales f
    ON r.sales_order_id = f.sales_order_id
   AND r.sales_order_detail_id = f.sales_order_detail_id
LEFT JOIN datawarehouse.dim_product p
    ON r.product_id = p.product_id
WHERE r.order_date >= '{period_start.strftime("%Y-%m-%d")}'
  AND r.order_date < '{period_end.strftime("%Y-%m-%d")}'
  AND f.sales_order_id IS NULL
ORDER BY r.sales_order_id, r.sales_order_detail_id ASC;
'''
df_fact_stage = pd.read_sql(sql=str_sql, con=db_datawarehouse)
print('Raw rows to transform into fact rows:', len(df_fact_stage))
df_fact_stage.head()

/tmp/ipykernel_48882/4034469977.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_fact_stage = pd.read_sql(sql=str_sql, con=db_datawarehouse)


Raw rows to transform into fact rows: 352


,sales_order_id,sales_order_detail_id,sales_order_number,customer_id,product_id,salesperson_id,order_date,ship_date,order_qty,unit_price,unit_price_discount,line_total,subtotal,tax_amt,freight,total_due,standard_cost
0,43659,1,SO43659,29825,776,279,2005-07-01,2005-07-08,1,2024.994,0.0,2024.994,20565.6206,1971.5149,616.0984,23153.2339,1898.0944
1,43659,2,SO43659,29825,777,279,2005-07-01,2005-07-08,3,2024.994,0.0,6074.982,20565.6206,1971.5149,616.0984,23153.2339,1898.0944
2,43659,3,SO43659,29825,778,279,2005-07-01,2005-07-08,1,2024.994,0.0,2024.994,20565.6206,1971.5149,616.0984,23153.2339,1898.0944
3,43659,4,SO43659,29825,771,279,2005-07-01,2005-07-08,1,2039.994,0.0,2039.994,20565.6206,1971.5149,616.0984,23153.2339,1912.1544
4,43659,5,SO43659,29825,772,279,2005-07-01,2005-07-08,1,2039.994,0.0,2039.994,20565.6206,1971.5149,616.0984,23153.2339,1912.1544


In [40]:
# Read dimension key maps from the warehouse

df_customer_map = pd.read_sql(
    'SELECT customer_key, customer_id FROM dim_store_customer;',
    con=db_datawarehouse
)

df_product_map = pd.read_sql(
    'SELECT product_key, product_id FROM dim_product;',
    con=db_datawarehouse
)

df_time_map = pd.read_sql(
    'SELECT time_key, full_date FROM dim_time;',
    con=db_datawarehouse
)
df_time_map['full_date'] = pd.to_datetime(df_time_map['full_date']).dt.date

df_salesperson_map = pd.read_sql(
    'SELECT salesperson_key, salesperson_id FROM dim_salesperson;',
    con=db_datawarehouse
)

# Transform
df_fact = df_fact_stage.copy()
df_fact['order_date'] = pd.to_datetime(df_fact['order_date']).dt.date
df_fact['ship_date'] = pd.to_datetime(df_fact['ship_date']).dt.date

df_fact = df_fact.merge(df_customer_map, on='customer_id', how='left')
df_fact = df_fact.merge(df_product_map, on='product_id', how='left')
df_fact = df_fact.merge(df_time_map, left_on='order_date', right_on='full_date', how='left')
df_fact = df_fact.merge(df_salesperson_map, on='salesperson_id', how='left')

# Row-level transformations
df_fact['gross_amount'] = (df_fact['order_qty'] * df_fact['unit_price']).round(4)
df_fact['discount_amount'] = (
    df_fact['order_qty'] * df_fact['unit_price'] * df_fact['unit_price_discount']
).round(4)
# df_fact['net_sales_amount'] = (df_fact['gross_amount'] - df_fact['discount_amount']).round(4)
df_fact['net_amount'] = df_fact['line_total']
df_fact['margin_amount'] = (df_fact['net_amount'] - (df_fact['order_qty'] * df_fact['standard_cost']).round(4)).round(4)

missing_customer = df_fact['customer_key'].isna().sum()
missing_product = df_fact['product_key'].isna().sum()
missing_time = df_fact['time_key'].isna().sum()
missing_salesperson = df_fact['salesperson_key'].isna().sum()

print('Missing customer keys   :', missing_customer)
print('Missing product keys    :', missing_product)
print('Missing time keys       :', missing_time)
print('Missing salesperson keys:', missing_salesperson)

if missing_customer or missing_product or missing_time or missing_salesperson:
    raise ValueError('One or more surrogate keys could not be resolved.')

df_fact = df_fact[[
    'sales_order_id',
    'sales_order_detail_id',
    'sales_order_number',
    'customer_key',
    'product_key',
    'time_key',
    'salesperson_key',
    'order_qty',
    'unit_price',
    'unit_price_discount',
    'gross_amount',
    'discount_amount',
    'margin_amount',
    'net_amount',
    'subtotal',
    'tax_amt',
    'freight',
    'total_due'
]].rename(columns={
    'time_key': 'order_date_key',
    'subtotal': 'header_subtotal',
    'tax_amt': 'header_tax_amt',
    'freight': 'header_freight',
    'total_due': 'header_total_due'
})

df_fact.head()

/tmp/ipykernel_48882/4131916407.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_customer_map = pd.read_sql(
/tmp/ipykernel_48882/4131916407.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_product_map = pd.read_sql(
/tmp/ipykernel_48882/4131916407.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_time_map = pd.read_sql(
/tmp/ipykernel_48882/4131916407.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 object

Missing customer keys   : 0
Missing product keys    : 0
Missing time keys       : 0
Missing salesperson keys: 0


,sales_order_id,sales_order_detail_id,sales_order_number,customer_key,product_key,order_date_key,salesperson_key,order_qty,unit_price,unit_price_discount,gross_amount,discount_amount,margin_amount,net_amount,header_subtotal,header_tax_amt,header_freight,header_total_due
0,43659,1,SO43659,24,44,20050701,6,1,2024.994,0.0,2024.994,0.0,126.8996,2024.994,20565.6206,1971.5149,616.0984,23153.2339
1,43659,2,SO43659,24,45,20050701,6,3,2024.994,0.0,6074.982,0.0,380.6988,6074.982,20565.6206,1971.5149,616.0984,23153.2339
2,43659,3,SO43659,24,46,20050701,6,1,2024.994,0.0,2024.994,0.0,126.8996,2024.994,20565.6206,1971.5149,616.0984,23153.2339
3,43659,4,SO43659,24,39,20050701,6,1,2039.994,0.0,2039.994,0.0,127.8396,2039.994,20565.6206,1971.5149,616.0984,23153.2339
4,43659,5,SO43659,24,40,20050701,6,1,2039.994,0.0,2039.994,0.0,127.8396,2039.994,20565.6206,1971.5149,616.0984,23153.2339


In [41]:
if len(df_fact) > 0:
    cursor = db_datawarehouse.cursor()
    for _, row in df_fact.iterrows():
        # Convert NaN to None for MySQL NULL compatibility
        row_values = tuple(None if pd.isna(val) else val for val in row)
        cursor.execute('''
            INSERT INTO fact_store_sales
            (sales_order_id, sales_order_detail_id, sales_order_number, customer_key, product_key,
             order_date_key, salesperson_key, order_qty, unit_price, unit_price_discount,
             gross_amount, discount_amount, margin_amount, net_amount, 
             header_subtotal, header_tax_amt, header_freight, header_total_due)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        ''', row_values)
    db_datawarehouse.commit()
    cursor.close()

pd.read_sql('SELECT * FROM fact_store_sales ORDER BY fact_store_sales_key LIMIT 10;', con=db_datawarehouse)

/tmp/ipykernel_48882/2236656236.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql('SELECT * FROM fact_store_sales ORDER BY fact_store_sales_key LIMIT 10;', con=db_datawarehouse)


,fact_store_sales_key,sales_order_id,sales_order_detail_id,sales_order_number,customer_key,product_key,order_date_key,salesperson_key,order_qty,unit_price,unit_price_discount,gross_amount,discount_amount,margin_amount,net_amount,header_subtotal,header_tax_amt,header_freight,header_total_due
0,1,43659,1,SO43659,24,44,20050701,6,1,2024.9940,0.0,2024.9940,0.0,126.8996,2024.9940,20565.6206,1971.5149,616.0984,23153.2339
1,2,43659,2,SO43659,24,45,20050701,6,3,2024.9940,0.0,6074.9820,0.0,380.6988,6074.9820,20565.6206,1971.5149,616.0984,23153.2339
2,3,43659,3,SO43659,24,46,20050701,6,1,2024.9940,0.0,2024.9940,0.0,126.8996,2024.9940,20565.6206,1971.5149,616.0984,23153.2339
3,4,43659,4,SO43659,24,39,20050701,6,1,2039.9940,0.0,2039.9940,0.0,127.8396,2039.9940,20565.6206,1971.5149,616.0984,23153.2339
4,5,43659,5,SO43659,24,40,20050701,6,1,2039.9940,0.0,2039.9940,0.0,127.8396,2039.9940,20565.6206,1971.5149,616.0984,23153.2339
5,6,43659,6,SO43659,24,41,20050701,6,2,2039.9940,0.0,4079.9880,0.0,255.6792,4079.9880,20565.6206,1971.5149,616.0984,23153.2339
6,7,43659,7,SO43659,24,42,20050701,6,1,2039.9940,0.0,2039.9940,0.0,127.8396,2039.9940,20565.6206,1971.5149,616.0984,23153.2339
7,8,43659,8,SO43659,24,7,20050701,6,3,28.8404,0.0,86.5212,0.0,-28.9557,86.5212,20565.6206,1971.5149,616.0984,23153.2339
8,9,43659,9,SO43659,24,9,20050701,6,1,28.8404,0.0,28.8404,0.0,-9.6519,28.8404,20565.6206,1971.5149,616.0984,23153.2339
9,10,43659,10,SO43659,24,3,20050701,6,6,5.7000,0.0,34.2000,0.0,13.8222,34.2000,20565.6206,1971.5149,616.0984,23153.2339


## 7. Validation checks

In [42]:

validation_sql = '''
SELECT 'dim_store_customer' AS table_name, COUNT(*) AS row_count FROM dim_store_customer
UNION ALL
SELECT 'dim_product' AS table_name, COUNT(*) AS row_count FROM dim_product
UNION ALL
SELECT 'dim_time' AS table_name, COUNT(*) AS row_count FROM dim_time
UNION ALL
SELECT 'dim_salesperson' AS table_name, COUNT(*) AS row_count FROM dim_salesperson
UNION ALL
SELECT 'raw_store_sales' AS table_name, COUNT(*) AS row_count FROM raw_store_sales
UNION ALL
SELECT 'fact_store_sales' AS table_name, COUNT(*) AS row_count FROM fact_store_sales;
'''
pd.read_sql(validation_sql, con=db_datawarehouse)


/tmp/ipykernel_48882/3625741796.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(validation_sql, con=db_datawarehouse)


,table_name,row_count
0,dim_store_customer,38
1,dim_product,46
2,dim_time,1
3,dim_salesperson,10
4,raw_store_sales,352
5,fact_store_sales,352


In [43]:

check_sql = '''
SELECT
    t.year_num,
    t.month_name,
    c.store_name,
    s.full_name AS salesperson,
    SUM(f.net_amount) AS total_sales
FROM fact_store_sales f
JOIN dim_store_customer c
    ON f.customer_key = c.customer_key
JOIN dim_product p
    ON f.product_key = p.product_key
JOIN dim_time t
    ON f.order_date_key = t.time_key
JOIN dim_salesperson s
    ON f.salesperson_key = s.salesperson_key
GROUP BY
    t.year_num,
    t.month_name,
    c.store_name,
    s.full_name
ORDER BY total_sales DESC
LIMIT 20;
'''
pd.read_sql(check_sql, con=db_datawarehouse)


/tmp/ipykernel_48882/2730343811.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(check_sql, con=db_datawarehouse)


,year_num,month_name,store_name,salesperson,total_sales
0,2005,July,Great Bikes,David R Campbell,42813.4333
1,2005,July,Sports Sales and Rental,Tsvi Michael Reiter,39373.7810
2,2005,July,Bike Dealers Association,Shu K Ito,38510.8973
3,2005,July,Retail Mall,José Edvaldo Saraiva,35944.1562
4,2005,July,Fitness Toy Store,Jillian Carson,33997.3702
5,2005,July,Original Bicycle Supply Company,José Edvaldo Saraiva,32726.4786
6,2005,July,"Health Spa, Limited",José Edvaldo Saraiva,28832.5289
7,2005,July,Capable Sales and Service,Pamela O Ansman-Wolfe,24432.6088
8,2005,July,Juvenile Sports Equipment,Tsvi Michael Reiter,20645.6340
9,2005,July,Better Bike Shop,Tsvi Michael Reiter,20565.6206


In [44]:

# Close database connections

db_adventureworks2012.close()
db_datawarehouse.close()

print('All database connections closed.')


All database connections closed.
